# L08 · Demonstration Acquisition and Scripted Experts

This lab applies the scripted-expert route selected in the lecture to one complete banana-to-bowl task:

```text
demonstration source choice → task and grasp contract → seven-phase expert
→ in-memory command/state trace → visual and containment evidence
```

The notebook keeps the controller in the shared package, but makes its phase structure, command schedule, measured response, and task result visible.

## Before you run

`ROBO_GENESIS_BACKEND=auto` selects the verified AMD backend when available and otherwise uses CPU. Set it to `cpu` to explicitly request the CPU fallback.

Starting with L08, `ROBO_GENESIS_RENDER` defaults to `1`: the normal learning path creates a world camera and displays the task stages. If rendering is unavailable, set it to `0` before starting the kernel; the rollout, schedule plot, command/state plot, and containment plot still run. Restart the kernel before changing either setting.

Predict first:

1. Which information belongs to `TaskSpec`, and which belongs to `GraspProfile`?
2. Why does settling occur before the seven action phases?
3. Why can commanded action and measured state both have shape `(T, 9)` without being equal?
4. Why must bowl containment check both horizontal placement and below-rim depth?

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np

from robo_genesis.build_scene import build_scene
from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.course_utils import environment_report, select_backend, to_numpy
from robo_genesis.grasp_demo import (
    GraspProfile,
    MOVE_MAX_DQ,
    MOVE_MIN_STEPS,
    TaskSpec,
    check_success,
    run_pick_place,
)
from robo_genesis.scene_config import WORLD_CAM_RES

lesson = load_course_manifest().lesson('L08')
assert lesson.slug == 'demonstration-acquisition-and-scripted-experts'
assert lesson.duration_minutes == 120
assert lesson.hardware.value == 'gpu-recommended'
assert lesson.status.value == 'planned'

backend_mode = os.environ.get('ROBO_GENESIS_BACKEND', 'auto').strip().lower()
if backend_mode not in {'auto', 'cpu'}:
    raise ValueError("ROBO_GENESIS_BACKEND must be 'auto' or 'cpu'")
render_value = os.environ.get('ROBO_GENESIS_RENDER', '1').strip()
if render_value not in {'0', '1'}:
    raise ValueError("ROBO_GENESIS_RENDER must be '0' or '1'")
render_enabled = render_value == '1'

environment = environment_report()

import genesis as gs

backend = gs.cpu if backend_mode == 'cpu' else select_backend(prefer_rocm=True)
gs.init(backend=backend, seed=0, precision='32', logging_level='warning')

if getattr(gs, 'amdgpu', None) is not None and gs.backend == gs.amdgpu:
    actual_backend = 'amdgpu'
elif gs.backend == gs.cpu:
    actual_backend = 'cpu'
else:
    actual_backend = str(gs.backend)

print(f'Run configuration: backend={actual_backend} (requested={backend_mode}), render={render_enabled}')


## Read the task and the seven-phase expert

`TaskSpec` states what should happen; `GraspProfile` states how the chosen object should be grasped. The diagram shows how the shared expert turns those two inputs into an observable sequence.

![The scripted expert settles the scene, then executes pregrasp, descend, grasp, lift, transport, release, and retreat in order.](../../docs/public/diagrams/l08-seven-phase-pick-place.svg)

The code below creates the concrete banana-to-bowl contract. The full motion implementation remains in `robo_genesis.grasp_demo`, so the notebook does not maintain a second controller.

In [ ]:
PHASES = (
    ('pregrasp', 'pose above object', 'IK + collision-checked plan'),
    ('descend', 'fixed-xy vertical approach', 'IK at descending z waypoints'),
    ('grasp', 'establish and hold contact', 'arm position + finger force'),
    ('lift', 'clear the tabletop', 'bounded-increment arm targets'),
    ('transport', 'move above the bowl', 'bounded-increment arm targets'),
    ('release', 'let the object enter the bowl', 'open finger position target'),
    ('retreat', 'move hand away and settle', 'direct retreat + settle'),
)
EXPECTED_FRAME_TAGS = (
    '00_start',
    '01_pregrasp',
    '02_reach',
    '03_grasp',
    '04_lift',
    '05_above_target',
    '06_release',
    '07_done',
)

task = TaskSpec(
    pick_object='011_banana',
    place_target='024_bowl',
    success_tol=0.06,
)
profile = task.grasp_profile()

assert tuple(row[0] for row in PHASES) == (
    'pregrasp', 'descend', 'grasp', 'lift', 'transport', 'release', 'retreat'
)
assert task.pick_object == '011_banana' and task.place_target == '024_bowl'
assert isinstance(profile, GraspProfile)
assert np.isfinite([profile.yaw_offset, profile.grasp_hand_z, profile.close_force]).all()

height_strategy = 'AABB-adaptive' if profile.center_align else 'fixed'
print(f'Task: {task.pick_object} → {task.place_target}; tolerance={task.success_tol:.3f} m')
print(
    f'Grasp profile: yaw offset={profile.yaw_offset:.1f}°, '
    f'height={height_strategy} ({profile.grasp_hand_z:.3f} m), '
    f'closing force={profile.close_force:.1f} N'
)


## See how the command bound changes a trajectory

During lift and transport, the expert interpolates from measured `q_start` to an IK goal. The number of waypoints is `max(MOVE_MIN_STEPS, ceil(Δq∞ / max_dq))`.

Before running the cell, predict what a smaller `CANDIDATE_MAX_DQ` will do. The two plots show the resulting target path and the actual infinity-norm increment between adjacent commands. They describe the command schedule—not measured acceleration or guaranteed grasp success.

In [ ]:
q_start = np.array([0.00, -0.30, 0.10, -1.80, 0.05, 1.55, 0.70], dtype=float)
q_goal = np.array([0.18, -0.08, 0.32, -1.42, -0.11, 1.82, 0.54], dtype=float)


def make_command_schedule(start, goal, max_dq):
    delta_q_inf = float(np.max(np.abs(goal - start)))
    waypoint_count = max(MOVE_MIN_STEPS, int(np.ceil(delta_q_inf / max_dq)))
    fractions = np.arange(1, waypoint_count + 1, dtype=float)[:, None] / waypoint_count
    commands = start + (goal - start) * fractions
    return np.vstack([start, commands])


delta_q_inf = float(np.max(np.abs(q_goal - q_start)))
waypoint_count = max(MOVE_MIN_STEPS, int(np.ceil(delta_q_inf / MOVE_MAX_DQ)))
baseline_schedule = make_command_schedule(q_start, q_goal, MOVE_MAX_DQ)

CANDIDATE_MAX_DQ = 0.003
candidate_schedule = make_command_schedule(q_start, q_goal, CANDIDATE_MAX_DQ)

baseline_steps = np.max(np.abs(np.diff(baseline_schedule, axis=0)), axis=1)
candidate_steps = np.max(np.abs(np.diff(candidate_schedule, axis=0)), axis=1)
baseline_max_step = float(np.max(baseline_steps))
candidate_max_step = float(np.max(candidate_steps))
candidate_waypoint_count = len(candidate_schedule) - 1

schedule_checks = {
    'baseline_endpoint': np.allclose(baseline_schedule[-1], q_goal),
    'baseline_step_bound': baseline_max_step <= MOVE_MAX_DQ + 1e-12,
    'candidate_endpoint': np.allclose(candidate_schedule[-1], q_goal),
    'candidate_step_bound': candidate_max_step <= CANDIDATE_MAX_DQ + 1e-12,
    'smaller_bound_uses_no_fewer_waypoints': candidate_waypoint_count >= waypoint_count,
}
failed_schedule = [name for name, passed in schedule_checks.items() if not passed]
if failed_schedule:
    raise AssertionError('Command-schedule checks failed: ' + ', '.join(failed_schedule))

joint_index = int(np.argmax(np.abs(q_goal - q_start)))
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(baseline_schedule[:, joint_index], label=f'max_dq={MOVE_MAX_DQ}')
axes[0].plot(candidate_schedule[:, joint_index], label=f'max_dq={CANDIDATE_MAX_DQ}')
axes[0].set_title(f'Joint {joint_index}: same goal, different schedules')
axes[0].set_xlabel('control step')
axes[0].set_ylabel('commanded target (rad)')
axes[0].legend()

axes[1].plot(np.arange(1, len(baseline_steps) + 1), baseline_steps, label='baseline')
axes[1].plot(np.arange(1, len(candidate_steps) + 1), candidate_steps, label='candidate')
axes[1].axhline(MOVE_MAX_DQ, color='C0', linestyle=':', alpha=0.8)
axes[1].axhline(CANDIDATE_MAX_DQ, color='C1', linestyle=':', alpha=0.8)
axes[1].set_title('Infinity-norm command increment')
axes[1].set_xlabel('control step')
axes[1].set_ylabel('max |Δq command| (rad)')
axes[1].legend()
figure.tight_layout()
plt.show()

print(f'Δq∞={delta_q_inf:.3f} rad')
print(f'{MOVE_MAX_DQ:.3f}: {waypoint_count} waypoints, max step={baseline_max_step:.4f} rad')
print(
    f'{CANDIDATE_MAX_DQ:.3f}: {candidate_waypoint_count} waypoints, '
    f'max step={candidate_max_step:.4f} rad'
)


## Build the fixed task scene

The shared builder reuses the L07 tabletop, Franka, YCB objects, controller configuration, and asset paths. This lesson keeps the task fixed with `scene_dr=None`; domain randomization belongs to L11.

The default path creates only the world camera needed for the stage montage. The explicit `ROBO_GENESIS_RENDER=0` fallback creates no camera but retains all numerical plots and task checks.

In [ ]:
bundle = build_scene(
    show_viewer=False,
    n_envs=1,
    add_world_cam=render_enabled,
    add_wrist_cam=False,
    add_video_cam=False,
    draw_world_frame=False,
    scene_dr=None,
)
initial_qpos = to_numpy(bundle.franka.get_qpos()).astype(float).reshape(-1)

build_checks = {
    'unbatched_franka_qpos': initial_qpos.shape == (9,),
    'franka_qpos_finite': np.isfinite(initial_qpos).all(),
    'task_entities_present': task.pick_object in bundle.ycb
    and task.place_target in bundle.ycb,
    'requested_world_camera': (bundle.world_cam is not None) == render_enabled,
    'no_wrist_camera': bundle.wrist_cam is None,
    'no_video_camera': bundle.video_cam is None,
}
failed_build = [name for name, passed in build_checks.items() if not passed]
if failed_build:
    raise AssertionError('Scene checks failed: ' + ', '.join(failed_build))

camera_summary = 'world camera ready' if render_enabled else 'no camera (explicit fallback)'
print(
    f'Scene ready: Franka qpos {initial_qpos.shape}, '
    f'{len(bundle.ycb)} named YCB objects, {camera_summary}.'
)


## Run one expert rollout

The lightweight recorder reads measured Franka qpos immediately before each corresponding command is sent and the simulator advances:

```text
state_t  = measured [arm_q(7), finger_q(2)]
action_t = commanded [arm_target(7), finger_target(2)]
```

Everything stays in memory. L09 will add timestamps, image sampling, episode boundaries, and persistent storage.

In [ ]:
class TraceRecorder:
    def __init__(self, scene_bundle):
        self.bundle = scene_bundle
        self.states = []
        self.actions = []

    def on_step(self, action):
        measured_qpos = to_numpy(self.bundle.franka.get_qpos()).astype(float).reshape(-1)
        commanded_action = to_numpy(action).astype(float).reshape(-1)
        self.states.append(measured_qpos.copy())
        self.actions.append(commanded_action.copy())


trace = TraceRecorder(bundle)
print('Running one banana-to-bowl scripted rollout ...')
rollout_success, stage_frames = run_pick_place(
    bundle,
    task,
    save_frames=render_enabled,
    recorder=trace,
)
state_trace = np.stack(trace.states)
action_trace = np.stack(trace.actions)

print(
    f'Rollout complete: {len(state_trace)} aligned state/action steps, '
    f'success={rollout_success}, stage images={len(stage_frames)}'
)


## Plot commanded targets against measured motion

Matching `(T, 9)` shapes mean the two sequences are aligned; they do not mean the values are equal. The first plot selects the arm joint with the largest observed difference and overlays its target and measured position. The second shows the largest arm tracking difference at each recorded step.

Read the curve as evidence of controller response, contact, and timing—not as a fixed result that every backend must reproduce exactly.

In [ ]:
trace_checks = {
    'same_nonzero_length': len(state_trace) == len(action_trace) > 0,
    'state_shape': state_trace.ndim == 2 and state_trace.shape[1] == 9,
    'action_shape': action_trace.ndim == 2 and action_trace.shape[1] == 9,
    'floating_arrays': np.issubdtype(state_trace.dtype, np.floating)
    and np.issubdtype(action_trace.dtype, np.floating),
    'finite_arrays': np.isfinite(state_trace).all() and np.isfinite(action_trace).all(),
}
tracking_error = action_trace - state_trace
trace_checks['command_state_difference_observed'] = bool(np.max(np.abs(tracking_error)) > 0.0)
failed_trace = [name for name, passed in trace_checks.items() if not passed]
if failed_trace:
    raise AssertionError('Trace checks failed: ' + ', '.join(failed_trace))

arm_error = np.abs(tracking_error[:, :7])
tracking_joint = int(np.argmax(np.max(arm_error, axis=0)))
steps = np.arange(len(state_trace))

figure, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(steps, action_trace[:, tracking_joint], '--', color='black', label='commanded target')
axes[0].plot(steps, state_trace[:, tracking_joint], color='C0', label='measured qpos')
axes[0].set_ylabel(f'joint {tracking_joint} (rad)')
axes[0].set_title('A command is a target; measured motion follows dynamically')
axes[0].legend()

axes[1].plot(steps, np.max(arm_error, axis=1), color='C3')
axes[1].set_xlabel('recorded control step')
axes[1].set_ylabel('max arm |command − state| (rad)')
axes[1].set_title('Tracking difference across the seven arm joints')
figure.tight_layout()
plt.show()

print(
    f'Trace: state={state_trace.shape}, action={action_trace.shape}; '
    f'largest observed arm difference={np.max(arm_error):.4f} rad '
    f'(joint {tracking_joint})'
)


## Visualize task completion

The final experiment first draws the two containment conditions even when camera rendering is disabled:

- top view: is the banana center inside the allowed bowl footprint?
- side view: is the banana AABB bottom below the rim-minus-margin threshold?

With the default render setting, it then displays the settled start and one world-view image after each expert phase. Detailed assertions stay silent unless something fails; successful execution ends with one `L08 CHECK: PASSED`.

In [ ]:
banana = bundle.ycb[task.pick_object]
bowl = bundle.ycb[task.place_target]
banana_pos = to_numpy(banana.get_pos()).astype(float).reshape(-1)
bowl_pos = to_numpy(bowl.get_pos()).astype(float).reshape(-1)
banana_aabb = to_numpy(banana.get_AABB()).astype(float).reshape(2, 3)
bowl_aabb = to_numpy(bowl.get_AABB()).astype(float).reshape(2, 3)

horizontal_distance = float(np.linalg.norm(banana_pos[:2] - bowl_pos[:2]))
bowl_rim_radius = 0.5 * float(
    min(bowl_aabb[1, 0] - bowl_aabb[0, 0], bowl_aabb[1, 1] - bowl_aabb[0, 1])
)
allowed_radius = min(task.success_tol, bowl_rim_radius)
within_footprint = horizontal_distance < allowed_radius

BOWL_RIM_MARGIN = 0.01
object_aabb_bottom_z = float(banana_aabb[0, 2])
bowl_rim_z = float(bowl_aabb[1, 2])
inside_bowl = object_aabb_bottom_z < bowl_rim_z - BOWL_RIM_MARGIN
shared_success = check_success(bundle, task)

outcome_checks = {
    'within_footprint': within_footprint,
    'inside_bowl': inside_bowl,
    'expanded_predicate_matches_shared_check':
    bool(within_footprint and inside_bowl) == shared_success,
    'rollout_result_matches_shared_check': bool(rollout_success) == shared_success,
    'task_completed': bool(rollout_success),
}

figure, axes = plt.subplots(1, 2, figsize=(11, 4.5))
allowed_circle = plt.Circle(
    bowl_pos[:2],
    allowed_radius,
    facecolor='C0',
    edgecolor='C0',
    alpha=0.18,
    label='allowed bowl footprint',
)
axes[0].add_patch(allowed_circle)
axes[0].scatter(*bowl_pos[:2], marker='+', s=140, color='C0', label='bowl center')
axes[0].scatter(*banana_pos[:2], marker='o', s=90, color='C1', label='banana center')
xy_pad = max(0.08, allowed_radius * 1.6, horizontal_distance * 1.2)
axes[0].set_xlim(bowl_pos[0] - xy_pad, bowl_pos[0] + xy_pad)
axes[0].set_ylim(bowl_pos[1] - xy_pad, bowl_pos[1] + xy_pad)
axes[0].set_aspect('equal')
axes[0].set_xlabel('world x (m)')
axes[0].set_ylabel('world y (m)')
axes[0].set_title(
    f'Horizontal: {horizontal_distance:.3f} m < {allowed_radius:.3f} m → {within_footprint}'
)
axes[0].legend(loc='best', fontsize=8)

threshold_z = bowl_rim_z - BOWL_RIM_MARGIN
axes[1].axhline(bowl_rim_z, color='C0', label='bowl rim')
axes[1].axhline(
    threshold_z,
    color='C0',
    linestyle='--',
    label='rim − margin',
)
axes[1].scatter(
    [0.0],
    [object_aabb_bottom_z],
    marker='o',
    s=90,
    color='C1',
    label='banana AABB bottom',
)
z_pad = 0.03
axes[1].set_xlim(-0.5, 0.5)
axes[1].set_ylim(
    min(object_aabb_bottom_z, threshold_z) - z_pad,
    max(object_aabb_bottom_z, bowl_rim_z) + z_pad,
)
axes[1].set_xticks([])
axes[1].set_ylabel('world z (m)')
axes[1].set_title(
    f'Vertical: {object_aabb_bottom_z:.3f} m < {threshold_z:.3f} m → {inside_bowl}'
)
axes[1].legend(loc='best', fontsize=8)
figure.suptitle('Bowl containment needs both conditions')
figure.tight_layout()
plt.show()

visual_status = 'SKIP — ROBO_GENESIS_RENDER=0; numerical plots remain available'
visual_checks = {
    'camera_absent_when_disabled': not render_enabled and bundle.world_cam is None,
    'frames_absent_when_disabled': not render_enabled and len(stage_frames) == 0,
}
if render_enabled:
    frame_tags = tuple(tag for tag, _ in stage_frames)
    frame_images = [to_numpy(image) for _, image in stage_frames]
    width, height = WORLD_CAM_RES
    visual_checks = {
        'world_camera_present': bundle.world_cam is not None,
        'eight_stage_frames': len(frame_images) == 8,
        'ordered_stage_tags': frame_tags == EXPECTED_FRAME_TAGS,
        'rgb_shapes': all(image.shape == (height, width, 3) for image in frame_images),
        'rgb_dtype': all(image.dtype == np.uint8 for image in frame_images),
        'finite_pixels': all(np.isfinite(image).all() for image in frame_images),
        'within_frame_variation': all(bool(np.std(image) > 0.0) for image in frame_images),
        'sequence_pixel_change': any(
            not np.array_equal(left, right)
            for left, right in zip(frame_images, frame_images[1:])
        ),
    }
    if all(visual_checks.values()):
        montage, axes = plt.subplots(2, 4, figsize=(16, 8))
        for axis, tag, image in zip(axes.ravel(), frame_tags, frame_images):
            axis.imshow(image)
            axis.set_title(tag)
            axis.axis('off')
        montage.suptitle('One observation after settling and after each expert phase')
        montage.tight_layout()
        plt.show()
        visual_status = 'start + seven phase images displayed'

final_checks = {
    'runtime_contract': environment['genesis_world'] == '1.3.3'
    and actual_backend in {'cpu', 'amdgpu'}
    and (backend_mode != 'cpu' or actual_backend == 'cpu'),
    'manifest_contract': lesson.status.value == 'planned'
    and lesson.duration_minutes == 120,
    'expert_contract': len(PHASES) == 7 and isinstance(profile, GraspProfile),
    'rate_schedule': all(schedule_checks.values()),
    'scene_build': all(build_checks.values()),
    'trace_evidence': all(trace_checks.values()),
    'outcome_evidence': all(outcome_checks.values()),
    'visual_branch': all(visual_checks.values()),
}
failed = [name for name, passed in final_checks.items() if not passed]
if failed:
    raise AssertionError('L08 checks failed: ' + ', '.join(failed))

print(
    f'Containment: horizontal={within_footprint} '
    f'({horizontal_distance:.4f} < {allowed_radius:.4f} m), '
    f'below rim={inside_bowl} '
    f'({object_aabb_bottom_z:.4f} < {threshold_z:.4f} m)'
)
print('Visual evidence:', visual_status)
print('L08 CHECK: PASSED')


## Checkpoint and connection to L09

Look back at your predictions. Using the phase diagram, command/state traces, and containment plots, you should now be able to explain:

- how `TaskSpec` separates the task definition from the object-specific grasp assumptions in `GraspProfile`;
- why settling prepares the episode before the seven expert actions begin;
- why commanded action and measured state can share shape `(T, 9)` while following different trajectories;
- why bowl containment requires both the horizontal-footprint and below-rim conditions.

This rollout shows that the scripted expert can complete the configured episode and that the recorder hook captures its command/state trace. Estimating repeatability requires multiple independent seeds.

For an optional motion diagnostic, run this command from the repository root, outside the notebook:

```bash
uv run python -m robo_genesis.tools.motion_probe --pick 011_banana --compare --seeds 0 1 2
```

For each seed, `--compare` resets both rollouts to the same initial state. It compares a direct-target baseline with the course's bounded-increment interpolation (`landed` in the output). Read `trEEacc` and `trJacc` as transport acceleration, `slip_mm` and `slip_deg` as in-gripper motion, and also compare `totSteps` and `succ`. The question is whether the changed motion schedule reduces acceleration or slip while preserving task completion from the same start—not simply which run uses fewer steps. Replace `011_banana` with `014_lemon` or `018_plum` to repeat the comparison with a different object shape and grasp profile.

L09 will connect the same recorder hook to a timed, persistent demonstration schema.